## B. Transformer

### 3.4 Self-Attention

> **做什么**：手写Self-Attention，理解Q/K/V注意力计算原理  
> **经典案例**：纯手工实现，不使用nn.Transformer


In [1]:
import torch
import torch.nn as nn
import math

device = torch.device('cpu')

# ====== 手写Self-Attention（理解原理） ======
class SelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        # 三个线性变换：生成Q(查询)、K(键)、V(值)
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        Q = self.W_q(x)  # 查询矩阵
        K = self.W_k(x)  # 键矩阵
        V = self.W_v(x)  # 值矩阵

        # 注意力分数 = Q*K^T / sqrt(d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_model)
        # Softmax归一化
        attn_weights = torch.softmax(scores, dim=-1)
        # 加权求和
        output = torch.matmul(attn_weights, V)
        return output, attn_weights

# 演示
attn = SelfAttention(d_model=64).to(device)
x = torch.randn(2, 10, 64).to(device)  # batch=2, seq_len=10, d=64
out, weights = attn(x)
print(f"输入形状: {x.shape}")
print(f"输出形状: {out.shape}")
print(f"注意力权重形状: {weights.shape}")   # (2, 10, 10)：10个token两两注意力
print(f"注意力权重行和: {weights[0,0].sum():.4f}")  # 约1.0（softmax归一化）

输入形状: torch.Size([2, 10, 64])
输出形状: torch.Size([2, 10, 64])
注意力权重形状: torch.Size([2, 10, 10])
注意力权重行和: 1.0000


### 3.5 Multi-Head Attention

> **做什么**：在Self-Attention基础上扩展为多头，各头关注不同子空间  
> **经典案例**：手写Multi-Head Attention

In [2]:
import torch
import torch.nn as nn
import math

device = torch.device('cpu')

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model必须能被n_heads整除"
        self.d_k = d_model // n_heads   # 每个头的维度
        self.n_heads = n_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)  # 输出投影

    def forward(self, x):
        B, S, D = x.shape
        # 生成Q,K,V并拆分为多头: (B, S, D) -> (B, n_heads, S, d_k)
        Q = self.W_q(x).view(B, S, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, S, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, S, self.n_heads, self.d_k).transpose(1, 2)

        # 每个头独立计算注意力
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, V)  # (B, n_heads, S, d_k)

        # 合并多头: (B, n_heads, S, d_k) -> (B, S, D)
        out = out.transpose(1, 2).contiguous().view(B, S, D)
        return self.W_o(out)

# 演示
mha = MultiHeadAttention(d_model=64, n_heads=8).to(device)
x = torch.randn(2, 10, 64).to(device)
out = mha(x)
print(f"多头注意力输出形状: {out.shape}")  # (2, 10, 64)

多头注意力输出形状: torch.Size([2, 10, 64])


### 3.6 BERT（使用transformers库做情感分类）

> **做什么**：用预训练BERT做文本情感分类（SST2）  
> **经典案例**：HuggingFace pipelines一行推理


In [1]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch

device = torch.device('cpu')

# ====== 方式1：一行推理（pipeline）======
sentiment = pipeline("sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=-1)
results = sentiment(["This movie is absolutely wonderful!",
                     "Terrible waste of time."])
for r in results:
    print(f"Label: {r['label']}, Confidence: {r['score']:.4f}")

# ====== 方式2：手动加载模型 ======
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)

text = "I love this product, it works perfectly!"
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
with torch.no_grad():
    logits = model(**inputs).logits
pred = torch.argmax(logits, dim=1).item()
print(f"Prediction: {'Positive' if pred==1 else 'Negative'}")

D:\JupyterNotebook\test\work2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
D:\JupyterNotebook\test\work2\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LAX\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In

Label: POSITIVE, Confidence: 0.9999
Label: NEGATIVE, Confidence: 0.9998


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 3906.85it/s]

Prediction: Positive


### 3.7 GPT（使用transformers库做文本生成）

> **做什么**：用预训练GPT-2做文本生成  
> **经典案例**：续写一段文本

In [2]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import torch

device = torch.device('cpu')

# ====== 方式1：pipeline一行生成 ======
generator = pipeline("text-generation", model="gpt2", device=-1)  # CPU
result = generator("Machine learning is", max_length=50, num_return_sequences=1)
print(f"Generated:\n{result[0]['generated_text']}")

# ====== 方式2：手动控制生成 ======
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)

prompt = "The future of artificial intelligence is"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

# 自回归生成
with torch.no_grad():
    output = model.generate(
        input_ids,
        max_length=60,        # 最大生成长度
        temperature=0.8,      # 温度：越低越确定，越高越随机
        top_k=50,             # Top-K采样
        do_sample=True
    )
generated = tokenizer.decode(output[0], skip_special_tokens=True)
print(f"Full generation:\n{generated}")

D:\JupyterNotebook\test\work2\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LAX\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4691.37it/s]
[transformers] Passing `generation_config` together with gener

Generated:
Machine learning is a powerful tool for solving complex problems. This is not to say that it's bad, but it is important to take the time to learn about it.

Learning to solve complex problems requires a lot of effort. In order to learn, you need to learn to solve problems. In our case, we are trying to solve an issue that we find is not yet solved.

That is why we are going to use the word "problem." Problem is a category of problems that are really difficult to solve, and that we are trying to solve. In other words, to solve an issue, you need to work on it.

Problem is not a category of problems that we can solve in the future. It is a category of problems that we can solve in the future. It's not a word that we can learn in a classroom.

Problem is a category of problems that we can learn in the future.

What's the difference between a problem and a problem-solving problem?

A problem is a problem that is solved. A problem is a problem that is solved.

A problem is a prob

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 3595.38it/s]


Full generation:
The future of artificial intelligence is now in doubt. At the moment of its birth, artificial intelligence is a matter of speculation and debate.

But the best way to understand it is to understand what it is and what it is not. That is what we have been doing for the last 40 years


### 3.8 ViT（简化版Vision Transformer）

> **做什么**：图像切成patch序列->Transformer编码->分类  
> **经典案例**：简化版ViT实现


In [5]:
import torch
import torch.nn as nn

device = torch.device('cpu')

class SimpleViT(nn.Module):
    def __init__(self, img_size=28, patch_size=4, in_channels=1,
                 d_model=64, n_heads=4, n_layers=3, n_classes=10):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2
        patch_dim = in_channels * patch_size ** 2

        # 1. Patch嵌入：将每个patch展平后线性映射到d_model维
        self.patch_embed = nn.Linear(patch_dim, d_model)
        # 2. 可学习的位置编码
        self.pos_embed = nn.Parameter(torch.randn(1, self.n_patches+1, d_model)*0.02)
        # 3. CLS token（分类用）
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model)*0.02)
        # 4. Transformer编码器
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            batch_first=True, dim_feedforward=128)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        # 5. 分类头
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, x):
        B = x.shape[0]
        # 切patch: (B, C, H, W) -> (B, n_patches, C*Hp*Wp)
        p = 4  # patch_size
        patches = x.unfold(2, p, p).unfold(3, p, p)          # (B, C, H/p, W/p, p, p)
        patches = patches.permute(0, 2, 3, 1, 4, 5).contiguous()  # (B, H/p, W/p, C, p, p)
        patches = patches.view(B, self.n_patches, -1)          # (B, 49, 16)

        # Patch嵌入 + 位置编码
        embeds = self.patch_embed(patches)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, embeds], dim=1) + self.pos_embed

        # Transformer编码
        x = self.encoder(x)

        # 用CLS token做分类
        return self.head(x[:, 0])

# 演示
vit = SimpleViT().to(device)
dummy_img = torch.randn(4, 1, 28, 28).to(device)  # 4张28x28灰度图
out = vit(dummy_img)
print(f"ViT output shape: {out.shape}")  # (4, 10)
print(f"Parameters: {sum(p.numel() for p in vit.parameters()):,}")

ViT output shape: torch.Size([4, 10])
Parameters: 105,418
